# COVID-19 lesion segmentation demo (inference only)

Text-prompted segmentation of 2D chest X-rays (MedCLIPSeg) and CT slices (MedSAM3), with the checkpoints selected on
the **validation** set (see `DEMO_README.md` and `demo/demo_config.yaml`). Research demo only: not for clinical diagnosis.

**Before running:**
1. A **GPU** runtime (T4 is enough). SAM3 needs CUDA.
2. A Hugging Face token from an account that has been granted access to the gated model `facebook/sam3`.
3. In Google Drive, in the folder set as `DRIVE_DIR` below:
   - `medclipseg_qata_best_dice.pth`, the fine-tuned MedCLIPSeg checkpoint
   - `medseg_demo_code.zip` (optional), the demo code made with `python demo/make_code_bundle.py`. If it is missing, the
     code is cloned from GitHub instead.

Then **Run all**. It pauses twice for you: the Google Drive authorization and the Hugging Face token. The last
cell keeps running (it is the demo server) and prints a public `gradio.live` link.

In [12]:
!nvidia-smi -L

GPU 0: Tesla T4 (UUID: GPU-b07ef85a-56e5-83e4-b8dd-11a6a70f571e)


## 1. Google Drive (checkpoint, and the code zip if you use one)

In [13]:
import os
from google.colab import drive
drive.mount("/content/drive")
DRIVE_DIR = "/content/drive/MyDrive/Colab Schoolwork/CPS 4881/medseg_demo_weights"
assert os.path.isdir(DRIVE_DIR), f"Folder not found: {DRIVE_DIR} - check the path (spelling, spaces, capitals)"
print(os.listdir(DRIVE_DIR))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
['medclipseg_qata_best_dice.pth', 'medseg_demo_code.zip']


## 2. Get the code

In [14]:
import zipfile
REPO = "/content/ind-study-covid-seg"
CODE_ZIP = os.path.join(DRIVE_DIR, "medseg_demo_code.zip")
if os.path.exists(CODE_ZIP):
    os.makedirs(REPO, exist_ok=True)
    zipfile.ZipFile(CODE_ZIP).extractall(REPO)  # overwrites older copies of the same files
    print("code from", CODE_ZIP)
elif not os.path.exists(REPO):
    !git clone --depth 1 https://github.com/nmea002/ind-study-covid-seg.git {REPO}
%cd {REPO}

code from /content/drive/MyDrive/Colab Schoolwork/CPS 4881/medseg_demo_weights/medseg_demo_code.zip
/content/ind-study-covid-seg


## 3. Hugging Face token (for the gated SAM3 base model)

In [15]:
# The token goes into HF_TOKEN, which huggingface_hub reads in the setup script and when the models load.
# Do not import huggingface_hub here: setup installs a different version, and a version already
# imported in this session would mix with the new one (ImportError: refresh_xet_connection_info).
# Leave it empty and press Enter if this runtime is already logged in (the saved token is used).
import getpass
if not os.environ.get("HF_TOKEN"):
    token = getpass.getpass("Hugging Face token (Enter = use the one saved on this runtime): ").strip()
    if token:
        os.environ["HF_TOKEN"] = token
    else:
        saved = os.path.exists(os.path.expanduser("~/.cache/huggingface/token"))
        print("using the saved token" if saved else "WARNING: no token saved on this runtime - SAM3 download will fail")

using the saved token


## 4. Install dependencies and download SAM3 + the released MedSAM3 LoRA (~5 min, once per runtime)

In [16]:
!bash demo/setup_demo_env.sh
# stop "Run all" here if any setup step failed, instead of failing confusingly later
assert _exit_code == 0, "setup failed - see the output above"

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for sam3-lora (pyproject.toml) ... done
/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
SAM3 checkpoint cached
MedSAM3 v1 LoRA: /content/ind-study-covid-seg/MedSAM3/medsam3_v1_lora.pt
medclipseg_qata_best_dice.pth
Done. Put the MedCLIPSeg checkpoint in demo/weights/, then: python demo/app.py --share


## 5. Copy the trained checkpoint from Drive

In [17]:
import shutil
os.makedirs("demo/weights", exist_ok=True)
src = os.path.join(DRIVE_DIR, "medclipseg_qata_best_dice.pth")
assert os.path.exists(src), f"medclipseg_qata_best_dice.pth is not in {DRIVE_DIR}"
if not os.path.exists("demo/weights/medclipseg_qata_best_dice.pth"):
    shutil.copy(src, "demo/weights/")
print(os.listdir("demo/weights"))

['medclipseg_qata_best_dice.pth']


## 6. Launch the demo

In [18]:
import sys
sys.path.insert(0, os.path.join(REPO, "demo"))
from app import Demo, build_ui

demo = Demo("demo/demo_config.yaml")
for modality in demo.modalities:  # load both checkpoints now, so the first request is fast
    print("loading", modality, "...")
    demo.backend(modality)
build_ui(demo).queue(default_concurrency_limit=1).launch(share=True, debug=True)

loading Chest X-ray ...
Loading UniMedCLIP (backbone: ViT-B/16)
Found checkpoint: checkpoints/unimed_clip_vit_b16.pt


Building custom UniMedCLIP
Turning off gradients in both the image and the text encoder
loading Chest CT slice ...
Replaced 61 nn.MultiheadAttention modules with MultiheadAttentionLoRA
Applied LoRA to 458 modules:
  - backbone.vision_backbone.trunk.blocks.0.attn.qkv
  - backbone.vision_backbone.trunk.blocks.0.attn.proj
  - backbone.vision_backbone.trunk.blocks.0.mlp.fc1
  - backbone.vision_backbone.trunk.blocks.0.mlp.fc2
  - backbone.vision_backbone.trunk.blocks.1.attn.qkv
  - backbone.vision_backbone.trunk.blocks.1.attn.proj
  - backbone.vision_backbone.trunk.blocks.1.mlp.fc1
  - backbone.vision_backbone.trunk.blocks.1.mlp.fc2
  - backbone.vision_backbone.trunk.blocks.2.attn.qkv
  - backbone.vision_backbone.trunk.blocks.2.attn.proj
  - backbone.vision_backbone.trunk.blocks.2.mlp.fc1
  - backbone.vision_backbone.trunk.blocks.2.mlp.fc2
  - backbone.vision_backbone.trunk.blocks.3.attn.qkv
  - backbone.vision_backbone.trunk.blocks.3.attn.proj
  - backbone.vision_backbone.trunk.blocks.3.ml

Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://22c6bc12fcc1f23c64.gradio.live
